# 07 — Fine-tune CLIP-L/14 on Flickr30K

Notebook này fine-tune model tốt nhất trong benchmark hiện tại:

```text
openai/clip-vit-large-patch14
```

Mục tiêu:

```text
Zero-shot CLIP-L/14 trên validation split
→ Fine-tune CLIP-L/14 trên train split
→ Evaluate lại Recall@1 / Recall@5 / Recall@10 trên validation split
```

Điểm quan trọng:

- Split theo **image**, không split theo caption.
- Mỗi ảnh có 5 caption, nhưng trong mỗi epoch train chỉ lấy ngẫu nhiên 1 caption/ảnh để giảm false negative trong contrastive loss.
- Mặc định train `projection_only` trước để an toàn với GPU.
- Có thể đổi sang `last_layers` sau khi pipeline chạy ổn.

## 1. Imports

In [1]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup

d:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config

Nếu bị CUDA out of memory, ưu tiên giảm:

```python
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
```

Nếu muốn chạy thử nhanh trước, đặt:

```python
MAX_IMAGES = 1000
EPOCHS = 1
```

In [2]:
# Notebook thường nằm trong folder notebooks/, nên project root là "..".
# Nếu bạn chạy notebook ngay tại project root, đoạn fallback bên dưới sẽ tự xử lý.
PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(".").resolve()

METADATA_PATH = PROJECT_ROOT / "data/processed/metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/clip_l14_finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "openai/clip-vit-large-patch14"
RUN_NAME = "clip_l14_projection_only"
RUN_DIR = OUTPUT_DIR / RUN_NAME
BEST_MODEL_DIR = RUN_DIR / "best_model"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# None = dùng toàn bộ Flickr30K. Đặt 1000 để smoke test nhanh.
MAX_IMAGES = None

# Split theo image.
SPLIT_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# Fine-tune modes:
# - "projection_only": freeze vision_model/text_model, train visual_projection/text_projection/logit_scale
# - "last_layers": train projection + last N transformer layers
# - "full": train toàn bộ model, rất nặng
FINETUNE_MODE = "projection_only"
NUM_LAST_LAYERS = 1

# Training config cho CLIP-L/14.
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
USE_AMP = True
MAX_GRAD_NORM = 1.0

# DataLoader config. Trên Windows/Colab để 0 thường ổn định hơn.
NUM_WORKERS = 0

# Evaluation config.
EVAL_EVERY_EPOCH = True
EVAL_IMAGE_BATCH_SIZE = 16
EVAL_TEXT_BATCH_SIZE = 64
K_VALUES = (1, 5, 10)

# Test evaluation chỉ nên bật sau khi đã chốt model tốt nhất.
RUN_TEST_EVAL = False

print(f"Project root: {PROJECT_ROOT}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Run dir: {RUN_DIR}")
print(f"Device: {DEVICE}")

Project root: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search
Metadata path: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\metadata.json
Output dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune
Run dir: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune\clip_l14_projection_only
Device: cuda


## 3. Reproducibility helpers

In [3]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


seed_everything(SPLIT_SEED)

## 4. Load and validate metadata

Notebook này dùng lại `metadata.json` từ notebook 01.

In [4]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy metadata: {METADATA_PATH}. "
        "Hãy chạy notebook 01_prepare_metadata.ipynb trước."
    )

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

if MAX_IMAGES is not None:
    metadata = metadata[:MAX_IMAGES]

required_keys = {"image_id", "image_path", "captions"}
for idx, item in enumerate(metadata):
    missing_keys = required_keys - set(item.keys())
    if missing_keys:
        raise ValueError(f"Item {idx} is missing keys: {missing_keys}")
    if len(item["captions"]) != 5:
        raise ValueError(f"Item {idx} does not have exactly 5 captions")
    image_path = PROJECT_ROOT / item["image_path"]
    if not image_path.exists():
        raise FileNotFoundError(image_path)

print(f"Number of images: {len(metadata):,}")
print(f"Number of captions: {sum(len(item['captions']) for item in metadata):,}")
print("Sample item:")
print(metadata[0])

Number of images: 31,782
Number of captions: 158,910
Sample item:
{'image_id': '1000092795.jpg', 'image_path': 'data/raw/Images/1000092795.jpg', 'captions': ['Two young guys with shaggy hair look at their hands while hanging out in the yard .', 'Two young , White males are outside near many bushes .', 'Two men in green shirts are standing in a yard .', 'A man in a blue shirt standing in a garden .', 'Two friends enjoy time spent together .']}


## 5. Split train / val / test by image

Không split theo caption, vì 5 caption của cùng một ảnh phải nằm cùng một split.

In [5]:
def split_metadata_by_image(metadata_items, train_ratio=0.8, val_ratio=0.1, seed=42):
    indices = np.arange(len(metadata_items))
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    n_total = len(indices)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    train_indices = indices[:n_train]
    val_indices = indices[n_train:n_train + n_val]
    test_indices = indices[n_train + n_val:]

    train_items = [metadata_items[i] for i in train_indices]
    val_items = [metadata_items[i] for i in val_indices]
    test_items = [metadata_items[i] for i in test_indices]

    return train_items, val_items, test_items


train_metadata, val_metadata, test_metadata = split_metadata_by_image(
    metadata,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SPLIT_SEED,
)

print(f"Train images: {len(train_metadata):,} | captions: {len(train_metadata) * 5:,}")
print(f"Val images:   {len(val_metadata):,} | captions: {len(val_metadata) * 5:,}")
print(f"Test images:  {len(test_metadata):,} | captions: {len(test_metadata) * 5:,}")

Train images: 25,425 | captions: 127,125
Val images:   3,178 | captions: 15,890
Test images:  3,179 | captions: 15,895


## 6. Dataset and collate function

Trong training, mỗi `__getitem__` chọn ngẫu nhiên 1 caption từ 5 caption của ảnh.

Lý do: nếu dùng cả 5 caption và shuffle theo pair, cùng một ảnh có thể xuất hiện nhiều lần trong cùng batch. Contrastive loss sẽ vô tình xem các caption đúng còn lại là negative.

In [6]:
class FlickrImageCaptionDataset(Dataset):
    def __init__(self, metadata_items, project_root, random_caption=True):
        self.metadata_items = metadata_items
        self.project_root = Path(project_root)
        self.random_caption = random_caption

    def __len__(self):
        return len(self.metadata_items)

    def __getitem__(self, idx):
        item = self.metadata_items[idx]
        image_path = self.project_root / item["image_path"]

        if self.random_caption:
            caption = random.choice(item["captions"])
        else:
            caption = item["captions"][0]

        return {
            "image_path": str(image_path),
            "caption": caption,
            "image_id": item["image_id"],
        }


def make_collate_fn(processor):
    def collate_fn(batch):
        images = []
        captions = []
        image_ids = []

        for sample in batch:
            with Image.open(sample["image_path"]) as img:
                images.append(img.convert("RGB"))
            captions.append(sample["caption"])
            image_ids.append(sample["image_id"])

        inputs = processor(
            text=captions,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs["image_ids"] = image_ids
        return inputs

    return collate_fn

## 7. Load CLIP-L/14 and choose trainable parameters

In [7]:
def load_clip_model_and_processor(model_name, device):
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)
    model = model.to(device)
    return model, processor


def set_requires_grad(module, requires_grad: bool):
    for param in module.parameters():
        param.requires_grad = requires_grad


def configure_trainable_parameters(model, mode="projection_only", num_last_layers=1):
    # Freeze all parameters first.
    set_requires_grad(model, False)

    # Always train projection heads and logit scale.
    set_requires_grad(model.visual_projection, True)
    set_requires_grad(model.text_projection, True)
    model.logit_scale.requires_grad = True

    if mode == "projection_only":
        pass

    elif mode == "last_layers":
        # Unfreeze last N transformer blocks.
        if num_last_layers <= 0:
            raise ValueError("NUM_LAST_LAYERS must be positive when mode='last_layers'")

        vision_layers = model.vision_model.encoder.layers
        text_layers = model.text_model.encoder.layers

        for layer in vision_layers[-num_last_layers:]:
            set_requires_grad(layer, True)
        for layer in text_layers[-num_last_layers:]:
            set_requires_grad(layer, True)

        # Also train final layer norms around the transformer outputs.
        if hasattr(model.vision_model, "post_layernorm"):
            set_requires_grad(model.vision_model.post_layernorm, True)
        if hasattr(model.text_model, "final_layer_norm"):
            set_requires_grad(model.text_model.final_layer_norm, True)

    elif mode == "full":
        set_requires_grad(model, True)

    else:
        raise ValueError(f"Unknown FINETUNE_MODE: {mode}")


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


model, processor = load_clip_model_and_processor(MODEL_NAME, DEVICE)
configure_trainable_parameters(model, FINETUNE_MODE, NUM_LAST_LAYERS)

total_params, trainable_params = count_parameters(model)
print(f"Model: {MODEL_NAME}")
print(f"Fine-tune mode: {FINETUNE_MODE}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable ratio: {trainable_params / total_params:.4%}")

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 18582.84it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: openai/clip-vit-large-patch14
Fine-tune mode: projection_only
Total parameters: 427,616,513
Trainable parameters: 1,376,257
Trainable ratio: 0.3218%


## 8. Embedding helpers and CLIP contrastive loss

Không dùng trực tiếp `model.get_image_features()` để tránh lỗi một số môi trường trả về `BaseModelOutputWithPooling`.

In [8]:
def l2_normalize(features):
    return features / features.norm(p=2, dim=-1, keepdim=True).clamp(min=1e-12)


def is_backbone_frozen(model):
    vision_trainable = any(p.requires_grad for p in model.vision_model.parameters())
    text_trainable = any(p.requires_grad for p in model.text_model.parameters())
    return (not vision_trainable) and (not text_trainable)


def get_projected_image_features(model, pixel_values, use_no_grad_backbone=False):
    if use_no_grad_backbone:
        with torch.no_grad():
            vision_outputs = model.vision_model(pixel_values=pixel_values)
            pooled_output = vision_outputs.pooler_output
    else:
        vision_outputs = model.vision_model(pixel_values=pixel_values)
        pooled_output = vision_outputs.pooler_output

    image_features = model.visual_projection(pooled_output)
    return l2_normalize(image_features)


def get_projected_text_features(model, input_ids, attention_mask=None, use_no_grad_backbone=False):
    if use_no_grad_backbone:
        with torch.no_grad():
            text_outputs = model.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            pooled_output = text_outputs.pooler_output
    else:
        text_outputs = model.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        pooled_output = text_outputs.pooler_output

    text_features = model.text_projection(pooled_output)
    return l2_normalize(text_features)


def compute_clip_loss(model, batch, use_no_grad_backbone=False):
    pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE, non_blocking=True)

    image_embeds = get_projected_image_features(
        model,
        pixel_values=pixel_values,
        use_no_grad_backbone=use_no_grad_backbone,
    )
    text_embeds = get_projected_text_features(
        model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_no_grad_backbone=use_no_grad_backbone,
    )

    logit_scale = model.logit_scale.exp().clamp(max=100)
    logits_per_image = logit_scale * image_embeds @ text_embeds.T
    logits_per_text = logits_per_image.T

    batch_size = image_embeds.shape[0]
    labels = torch.arange(batch_size, device=DEVICE)

    loss_i2t = F.cross_entropy(logits_per_image, labels)
    loss_t2i = F.cross_entropy(logits_per_text, labels)
    loss = (loss_i2t + loss_t2i) / 2

    return loss

## 9. Evaluation: Recall@K on a metadata split

Evaluation dùng toàn bộ 5 caption của mỗi ảnh trong split.

In [9]:
def build_captions_and_gt(metadata_items):
    captions = []
    gt_indices = []

    for image_index, item in enumerate(metadata_items):
        for caption in item["captions"]:
            captions.append(caption)
            gt_indices.append(image_index)

    return captions, np.array(gt_indices, dtype=np.int64)


@torch.no_grad()
def encode_images_for_eval(model, processor, metadata_items, batch_size, device):
    model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(metadata_items), batch_size), desc="Encoding eval images"):
        batch_items = metadata_items[start:start + batch_size]
        images = []

        for item in batch_items:
            image_path = PROJECT_ROOT / item["image_path"]
            with Image.open(image_path) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        image_features = get_projected_image_features(
            model,
            pixel_values=pixel_values,
            use_no_grad_backbone=False,
        )
        all_embeddings.append(image_features.cpu().numpy().astype(np.float32))

    return np.concatenate(all_embeddings, axis=0)


@torch.no_grad()
def evaluate_recall_at_k_on_split(
    model,
    processor,
    metadata_items,
    image_batch_size,
    text_batch_size,
    device,
    k_values=(1, 5, 10),
):
    model.eval()

    captions, gt_indices = build_captions_and_gt(metadata_items)
    image_embeddings = encode_images_for_eval(
        model=model,
        processor=processor,
        metadata_items=metadata_items,
        batch_size=image_batch_size,
        device=device,
    )

    image_emb_tensor = torch.from_numpy(image_embeddings).float().to(device)
    max_k = max(k_values)
    hit_counts = {k: 0 for k in k_values}
    total = 0

    for start in tqdm(range(0, len(captions), text_batch_size), desc="Evaluating Recall@K"):
        end = min(start + text_batch_size, len(captions))
        batch_texts = captions[start:end]
        batch_gt = gt_indices[start:end]

        inputs = processor(
            text=batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        text_features = get_projected_text_features(
            model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_no_grad_backbone=False,
        )

        similarity = text_features @ image_emb_tensor.T
        topk_indices = torch.topk(similarity, k=max_k, dim=1).indices.cpu().numpy()

        for k in k_values:
            topk = topk_indices[:, :k]
            correct = np.any(topk == batch_gt[:, None], axis=1)
            hit_counts[k] += int(correct.sum())

        total += len(batch_texts)

    recalls = {f"Recall@{k}": hit_counts[k] / total for k in k_values}
    recalls["num_images"] = len(metadata_items)
    recalls["num_queries"] = len(captions)
    return recalls

## 10. Build train DataLoader

In [10]:
train_dataset = FlickrImageCaptionDataset(
    metadata_items=train_metadata,
    project_root=PROJECT_ROOT,
    random_caption=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=make_collate_fn(processor),
)

print(f"Train steps per epoch: {len(train_loader):,}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

Train steps per epoch: 3,178
Effective batch size: 32


## 11. Zero-shot evaluation on validation split

Đây là baseline công bằng cho model trước fine-tune, trên đúng validation split.

In [11]:
zero_shot_result_path = RUN_DIR / "zero_shot_val_results.json"

if zero_shot_result_path.exists():
    print(f"Loading existing zero-shot results from: {zero_shot_result_path}")
    with zero_shot_result_path.open("r", encoding="utf-8") as f:
        zero_shot_val_results = json.load(f)
else:
    zero_shot_val_results = evaluate_recall_at_k_on_split(
        model=model,
        processor=processor,
        metadata_items=val_metadata,
        image_batch_size=EVAL_IMAGE_BATCH_SIZE,
        text_batch_size=EVAL_TEXT_BATCH_SIZE,
        device=DEVICE,
        k_values=K_VALUES,
    )
    zero_shot_val_results["model_name"] = MODEL_NAME
    zero_shot_val_results["split"] = "val"
    zero_shot_val_results["stage"] = "zero_shot"

    with zero_shot_result_path.open("w", encoding="utf-8") as f:
        json.dump(zero_shot_val_results, f, indent=2)

zero_shot_val_results

Evaluating Recall@K: 100%|██████████| 249/249 [00:12<00:00, 19.76it/s]


{'Recall@1': 0.5271869100062933,
 'Recall@5': 0.7849590937696664,
 'Recall@10': 0.8605412208936438,
 'num_images': 3178,
 'num_queries': 15890,
 'model_name': 'openai/clip-vit-large-patch14',
 'split': 'val',
 'stage': 'zero_shot'}

## 12. Optimizer and scheduler

In [12]:
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = num_update_steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE == "cuda")
use_no_grad_backbone = is_backbone_frozen(model)

print(f"Total update steps: {total_training_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")
print(f"Use AMP: {USE_AMP and DEVICE == 'cuda'}")
print(f"Use no_grad for frozen backbone: {use_no_grad_backbone}")

Total update steps: 2,385
Warmup steps: 119
Use AMP: True
Use no_grad for frozen backbone: True


C:\Users\Admin\AppData\Local\Temp\ipykernel_26548\2161114482.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE == "cuda")


## 13. Training loop

Mặc định notebook sẽ evaluate validation sau mỗi epoch và lưu checkpoint tốt nhất theo `Recall@5`.

In [13]:
def set_training_mode(model, finetune_mode):
    if finetune_mode == "projection_only":
        model.eval()
        model.visual_projection.train()
        model.text_projection.train()
        # logit_scale là parameter trực tiếp, không cần .train()
    else:
        model.train()


history = []
best_recall_at_5 = -1.0
best_epoch = None

start_train_time = time.time()
global_update_step = 0

for epoch in range(1, EPOCHS + 1):
    set_training_mode(model, FINETUNE_MODE)
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    num_loss_items = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(progress_bar, start=1):
        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE == "cuda"):
            loss = compute_clip_loss(
                model=model,
                batch=batch,
                use_no_grad_backbone=use_no_grad_backbone,
            )
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        running_loss += float(loss.detach().cpu())
        num_loss_items += 1

        should_update = (step % GRAD_ACCUM_STEPS == 0) or (step == len(train_loader))
        if should_update:
            if MAX_GRAD_NORM is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_update_step += 1

        avg_loss = running_loss / max(num_loss_items, 1)
        progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": scheduler.get_last_lr()[0]})

    epoch_train_loss = running_loss / max(num_loss_items, 1)
    epoch_record = {
        "epoch": epoch,
        "train_loss": epoch_train_loss,
        "global_update_step": global_update_step,
    }

    if EVAL_EVERY_EPOCH:
        clear_memory()
        val_results = evaluate_recall_at_k_on_split(
            model=model,
            processor=processor,
            metadata_items=val_metadata,
            image_batch_size=EVAL_IMAGE_BATCH_SIZE,
            text_batch_size=EVAL_TEXT_BATCH_SIZE,
            device=DEVICE,
            k_values=K_VALUES,
        )
        epoch_record.update({f"val_{k}": v for k, v in val_results.items() if k.startswith("Recall@")})

        current_recall_at_5 = val_results["Recall@5"]
        print(f"Epoch {epoch} validation results:", val_results)

        if current_recall_at_5 > best_recall_at_5:
            best_recall_at_5 = current_recall_at_5
            best_epoch = epoch
            BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(BEST_MODEL_DIR)
            processor.save_pretrained(BEST_MODEL_DIR)

            best_payload = {
                "model_name": MODEL_NAME,
                "run_name": RUN_NAME,
                "finetune_mode": FINETUNE_MODE,
                "best_epoch": best_epoch,
                "best_recall_at_5": best_recall_at_5,
                "val_results": val_results,
                "config": {
                    "batch_size": BATCH_SIZE,
                    "grad_accum_steps": GRAD_ACCUM_STEPS,
                    "epochs": EPOCHS,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "warmup_ratio": WARMUP_RATIO,
                    "num_last_layers": NUM_LAST_LAYERS,
                    "use_amp": USE_AMP,
                    "split_seed": SPLIT_SEED,
                },
            }
            with (RUN_DIR / "best_val_results.json").open("w", encoding="utf-8") as f:
                json.dump(best_payload, f, indent=2)

            print(f"Saved new best model to: {BEST_MODEL_DIR}")

    history.append(epoch_record)

    with (RUN_DIR / "training_history.json").open("w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

train_elapsed = time.time() - start_train_time
print(f"Training finished in {train_elapsed:.2f} seconds")
print(f"Best epoch: {best_epoch} | Best Recall@5: {best_recall_at_5:.4f}")

Epoch 1/3:   0%|          | 0/3178 [00:00<?, ?it/s]C:\Users\Admin\AppData\Local\Temp\ipykernel_26548\66495091.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE == "cuda"):
Evaluating Recall@K: 100%|██████████| 249/249 [00:12<00:00, 19.78it/s]


Epoch 1 validation results: {'Recall@1': 0.595531780994336, 'Recall@5': 0.8416614222781623, 'Recall@10': 0.9025802391441158, 'num_images': 3178, 'num_queries': 15890}


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


Saved new best model to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune\clip_l14_projection_only\best_model


Evaluating Recall@K: 100%|██████████| 249/249 [00:12<00:00, 19.91it/s]


...

Saved new best model to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune\clip_l14_projection_only\best_model


Evaluating Recall@K: 100%|██████████| 249/249 [00:11<00:00, 21.08it/s]


Epoch 3 validation results: {'Recall@1': 0.6049087476400252, 'Recall@5': 0.8458779106356199, 'Recall@10': 0.9067967275015734, 'num_images': 3178, 'num_queries': 15890}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Saved new best model to: D:\CITD\HK3\Python for ML\Clip_Retrieval\clip-faiss-search\data\processed\clip_l14_finetune\clip_l14_projection_only\best_model
Training finished in 1285.47 seconds
Best epoch: 3 | Best Recall@5: 0.8459


## 14. Load best checkpoint and evaluate again on validation split

In [14]:
if BEST_MODEL_DIR.exists():
    clear_memory()
    best_model = CLIPModel.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
    best_processor = CLIPProcessor.from_pretrained(BEST_MODEL_DIR)
else:
    print("Best model directory does not exist. Using current model instead.")
    best_model = model
    best_processor = processor

fine_tuned_val_results = evaluate_recall_at_k_on_split(
    model=best_model,
    processor=best_processor,
    metadata_items=val_metadata,
    image_batch_size=EVAL_IMAGE_BATCH_SIZE,
    text_batch_size=EVAL_TEXT_BATCH_SIZE,
    device=DEVICE,
    k_values=K_VALUES,
)
fine_tuned_val_results["model_name"] = MODEL_NAME
fine_tuned_val_results["split"] = "val"
fine_tuned_val_results["stage"] = "fine_tuned"
fine_tuned_val_results["finetune_mode"] = FINETUNE_MODE

with (RUN_DIR / "fine_tuned_val_results.json").open("w", encoding="utf-8") as f:
    json.dump(fine_tuned_val_results, f, indent=2)

fine_tuned_val_results

Evaluating Recall@K: 100%|██████████| 249/249 [00:12<00:00, 19.90it/s]


{'Recall@1': 0.6049087476400252,
 'Recall@5': 0.8458779106356199,
 'Recall@10': 0.9067967275015734,
 'num_images': 3178,
 'num_queries': 15890,
 'model_name': 'openai/clip-vit-large-patch14',
 'split': 'val',
 'stage': 'fine_tuned',
 'finetune_mode': 'projection_only'}

## 15. Compare zero-shot vs fine-tuned

In [15]:
comparison_rows = []
for result in [zero_shot_val_results, fine_tuned_val_results]:
    comparison_rows.append({
        "stage": result["stage"],
        "split": result["split"],
        "Recall@1": result["Recall@1"],
        "Recall@5": result["Recall@5"],
        "Recall@10": result["Recall@10"],
        "num_images": result["num_images"],
        "num_queries": result["num_queries"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["Recall@1 (%)"] = comparison_df["Recall@1"] * 100
comparison_df["Recall@5 (%)"] = comparison_df["Recall@5"] * 100
comparison_df["Recall@10 (%)"] = comparison_df["Recall@10"] * 100

comparison_df.to_csv(RUN_DIR / "zero_shot_vs_finetuned_val.csv", index=False)
comparison_df

,stage,split,Recall@1,Recall@5,Recall@10,num_images,num_queries,Recall@1 (%),Recall@5 (%),Recall@10 (%)
0,zero_shot,val,0.527187,0.784959,0.860541,3178,15890,52.718691,78.495909,86.054122
1,fine_tuned,val,0.604909,0.845878,0.906797,3178,15890,60.490875,84.587791,90.679673


## 16. Optional: test evaluation

Chỉ bật `RUN_TEST_EVAL = True` khi đã chốt cấu hình tốt nhất. Test split không nên dùng để tuning.

In [16]:
if RUN_TEST_EVAL:
    test_results = evaluate_recall_at_k_on_split(
        model=best_model,
        processor=best_processor,
        metadata_items=test_metadata,
        image_batch_size=EVAL_IMAGE_BATCH_SIZE,
        text_batch_size=EVAL_TEXT_BATCH_SIZE,
        device=DEVICE,
        k_values=K_VALUES,
    )
    test_results["model_name"] = MODEL_NAME
    test_results["split"] = "test"
    test_results["stage"] = "fine_tuned"
    test_results["finetune_mode"] = FINETUNE_MODE

    with (RUN_DIR / "fine_tuned_test_results.json").open("w", encoding="utf-8") as f:
        json.dump(test_results, f, indent=2)

    display(test_results)
else:
    print("RUN_TEST_EVAL is False. Skipping test evaluation.")

RUN_TEST_EVAL is False. Skipping test evaluation.


## 17. Notes for next experiments

Sau khi notebook này chạy ổn, có 2 hướng tiếp theo:

1. Đổi `FINETUNE_MODE = "last_layers"` và thử `NUM_LAST_LAYERS = 1` hoặc `2`.
2. Tạo notebook riêng để benchmark SigLIP2 zero-shot trên cùng split, ví dụ:

```text
08_benchmark_siglip2_zero_shot.ipynb
```

Không nên trộn SigLIP2 vào notebook này vì processor, model output và loss khác CLIP.